# 10 — Manipulation stratégique et règles à comité

**Navigation** : [<< 09-Committees-STV-Monroe-ChamberlinCourant.ipynb](09-Committees-STV-Monroe-ChamberlinCourant.ipynb) | [Index](../README.md)

*Détecter, mesurer et modéliser la manipulation stratégique sous les règles de scrutin à comité (pluralité, Borda, Chamberlin-Courant, Monroe, STV).*

## Ce que ce carnet démontre

Un votant qui ment sur ses préférences peut améliorer son sort. La question n'est donc pas de savoir *si* c'est possible, mais **quand** et **à quel prix**. Ce carnet :

1. **reproduit** le témoin de l'Exemple 6.1 du *Handbook of Computational Social Choice* (chapitre 6, p. 127) ;
2. **automatise** la recherche de toute déviation gagnante, et exhibe sur ce même profil les **quatre** déviations que le manuel ne cite pas toutes ;
3. **vérifie** les trois branches du théorème de Gibbard-Satterthwaite (Théorème 6.2, p. 128) sur l'espace exhaustif des 216 profils à trois votants et trois candidats ;
4. **mesure** la fréquence de manipulation, en fonction de la règle et de la taille du comité, sur deux espaces énumérés exhaustivement ;
5. **encode** la recherche d'une déviation gagnante en satisfiabilité (Z3) et **confronte** le solveur à l'énumération brute.

## Prérequis

- Python 3.10+ (`z3-solver` pour la section 8) ;
- aucun GPU, aucun formaliseur : le carnet tourne sur CPU en quelques minutes.

## Plan

| Section | Contenu |
|---|---|
| 1 | Cadre : profil, bulletin, utilité *best-in-W* |
| 2 | Pluralité et le témoin de l'Exemple 6.1 |
| 3 | Les cinq règles de scrutin à comité |
| 4 | Le détecteur de manipulation |
| 5 | Les trois branches du Théorème 6.2 |
| 6 | Fréquence de manipulation (énumération exhaustive) |
| 7 | Un espace plus large : quatre candidats |
| 8 | Z3 : chercher la déviation symboliquement |
| 9 | Exercices |
| 10 | Conclusion et références |

## 1. Cadre : profil, bulletin, utilité

Une **élection** est décrite par un *profil* `R`, liste de bulletins : chaque bulletin est un classement complet des `m` candidats, du préféré au moins préféré. Un **comité** `W` est un sous-ensemble de `k` candidats.

L'**utilité** d'un votant pour un comité est celle de son **meilleur membre** (*best-in-W*) : si `W` est non vide et que le meilleur candidat de `W` pour ce votant est à la position `t` dans son bulletin, l'utilité vaut `m - t` (échelle de Borda). L'utilité est nulle si `W` est vide.

Trois conventions sont fixées dès maintenant et ne changeront plus :

- **départage** : à égalité, le candidat **alphabétiquement le plus petit** est retenu (convention du manuel pour la pluralité) ;
- **comparaison de comités** : deux comités sont comparés comme des **ensembles** — l'ordre d'élimination (pour le STV) ne fait pas partie du résultat ;
- **complétion du STV** : le STV élit un candidat dès qu'il atteint le quota de Droop, et n'élimine le moins bien classé que tant qu'il reste assez de **survivants** pour occuper les sièges encore vacants ; le comité est ensuite complété par les **survivants**, classés par votes décroissants puis par ordre alphabétique. Un candidat **éliminé** ne peut pas être réintégré : tous les éliminés ont un score nul, les classer entre eux reviendrait à les départager alphabétiquement au lieu de les classer par popularité. Le comité rendu a donc toujours exactement `k` membres, et le résultat est un ensemble canonique (trié).

Le fait que l'utilité soit *best-in-W* a une conséquence structurante : un votant n'est pas « attaché » à un candidat précis, il est attaché au **meilleur** membre du comité. C'est ce qui rend les règles proportionnelles sensibles à la manipulation : déplacer le comité d'un cran suffit à changer le premier choix d'un votant.


In [1]:
from collections import Counter
from itertools import combinations, permutations, product
import random
import time

T0 = time.monotonic()


def rang(bulletin, c):
    """Position d'un candidat dans un bulletin (1 = préféré)."""
    return bulletin.index(c) + 1


def score_borda(bulletin, W, m):
    """Score Borda du meilleur membre de W pour ce bulletin (0 si W est vide)."""
    positions = [rang(bulletin, c) for c in W if c in bulletin]
    if not positions:
        return 0
    return m - min(positions)


def utilite(bulletin, W, m):
    """Utilité du votant pour le comité W : rang de son meilleur membre (best-in-W)."""
    return score_borda(bulletin, W, m)


print("rang de 'b' dans (a,b,c)        :", rang(('a', 'b', 'c'), 'b'))
print("score Borda de (b,c) sur (a,b,c) :", score_borda(('a', 'b', 'c'), ('b', 'c'), 3))
print("utilité de (a,b,c) pour (b,c)    :", utilite(('a', 'b', 'c'), ('b', 'c'), 3))
print(f"[temps] {time.monotonic() - T0:.2f}s")


rang de 'b' dans (a,b,c)        : 2
score Borda de (b,c) sur (a,b,c) : 1
utilité de (a,b,c) pour (b,c)    : 1
[temps] 0.00s


## 2. Pluralité et le témoin de l'Exemple 6.1

Le chapitre 6 du *Handbook of Computational Social Choice* ouvre par un exemple minimal. Le texte (p. 127) :

> **Example 6.1.** *Consider an election with three alternatives, a, b, and c, and three voters, 1, 2, and 3. Suppose the rule used is plurality — an alternative gets a point each time it is ranked first by a voter, and the alternative with the most points wins — with ties broken toward alternatives earlier in the alphabet. Suppose voter 3 knows (or strongly suspects) that voter 1 will rank a first in her vote, and that voter 2 will rank b first. Voter 3's true preferences are c b a. If she votes truthfully, this will result in a three-way tie, broken in favor of a which is 3's least preferred alternative. If, instead, voter 3 ranks b first, then b will win instead. Hence, voter 3 has an incentive to cast a vote that does not reflect her true preferences.*

Trois points méritent d'être isolés :

- la règle est la **pluralité** : un point par première position ;
- le départage se fait **par ordre alphabétique** — c'est lui qui fait perdre à la votante 3 son pire scénario ;
- la *manipulation* est ici définie comme un **écart entre le bulletin déclaré et les préférences vraies**, sans hypothèse de rationalité ni d'information parfaite.

Reconstruisons ce profil et vérifions le témoin, avant d'automatiser quoi que ce soit.


In [2]:
def pluralite(R, k, m=None):
    """Pluralité top-k : les k candidats les plus votés en première position.
    Départage alphabétique à égalité de points (convention de l'Exemple 6.1)."""
    cands = sorted(set().union(*R))
    voix = Counter(r[0] for r in R)
    return tuple(sorted(cands, key=lambda c: (-voix[c], c))[:k])


# Profil de l'Exemple 6.1 : v1 classe a en tête, v2 classe b en tête,
# v3 (préférences vraies c > b > a) classe c en tête -> match nul à trois.
R61 = [('a', 'b', 'c'), ('b', 'c', 'a'), ('c', 'b', 'a')]
R61_faux = list(R61)
R61_faux[2] = ('b', 'c', 'a')          # la votante 3 déclare b en tête

W_honnete = pluralite(R61, 1, 3)
W_manipule = pluralite(R61_faux, 1, 3)

print("bulletins sincères         :", [''.join(r) for r in R61])
print("points sous vote sincère   :", dict(Counter(r[0] for r in R61)))
print("comité sous vote sincère   :", W_honnete,
      "- utilité de la votante 3 :", utilite(R61[2], W_honnete, 3))
print("comité si v3 déclare b>c>a :", W_manipule,
      "- utilité de la votante 3 :", utilite(R61[2], W_manipule, 3))


bulletins sincères         : ['abc', 'bca', 'cba']
points sous vote sincère   : {'a': 1, 'b': 1, 'c': 1}
comité sous vote sincère   : ('a',) - utilité de la votante 3 : 0
comité si v3 déclare b>c>a : ('b',) - utilité de la votante 3 : 1


### Lecture du résultat

Le vote sincère produit bien un **match nul à trois** (un point chacun) : le comité est `('a',)`, par départage alphabétique. Pour la votante 3, `a` est son **dernier** choix — l'utilité vaut 0. En déclarant `b > c > a`, elle fait basculer l'élection sur `b`, où elle obtient son **deuxième** choix : l'utilité passe de 0 à 1. Le gain est faible — un rang — mais il est strictement positif, et c'est **suffisant** pour qualifier le profil de manipulable.

Le manuel s'arrête sur ce profil. Il ne dit pas si d'autres déviations existent, ni à quelle fréquence la manipulation apparaît sur l'ensemble des profils. C'est ce que le carnet mesure.

## 3. Les cinq règles de scrutin à comité

Cinq règles sont implémentées dans ce carnet, toutes sous la même signature `regle(R, k, m) -> comité` :

| Règle | Principe |
|---|---|
| Pluralité (top-k) | les `k` candidats les plus votés en première position |
| Borda (top-k) | les `k` candidats de plus haut score Borda total |
| Chamberlin-Courant | le comité qui **maximise la somme des utilités best-in-W** |
| Monroe | le comité maximisant la satisfaction sous **quotas équilibrés** par élu |
| STV (quota de Droop) | élimination et transfert de surplus, quota de Droop `n // (k+1) + 1` |

**Copie pédagogique déclarée** : ces cinq règles sont une réimplémentation, dans ce carnet, des règles du carnet jumeau `09-Committees-STV-Monroe-ChamberlinCourant.ipynb`. *Motif : le dépôt ne contient aucun module de choix social réutilisable en Python ; ces règles sont réécrites ici pour que le détecteur de manipulation reste lisible et mesurable, et pour disposer d'une base de comparaison documentée entre le carnet et son jumeau.* Deux **adaptations** sont signalées dans les docstrings : la signature uniforme `(R, k, m)`, et la garantie de **résoluté** du STV (le comité a toujours `k` membres) — exigence du Théorème 6.2, qui ne porte que sur des règles **résolues**.


In [3]:
def borda_topk(R, k, m=None):
    """Borda top-k : les k candidats de plus haut score Borda total.
    Départage alphabétique à égalité de score."""
    if m is None:
        m = len(R[0])
    cands = sorted(set().union(*R))
    scores = {c: sum(m - rang(r, c) for r in R) for c in cands}
    return tuple(sorted(cands, key=lambda c: (-scores[c], c))[:k])


def cc(R, k, m=None):
    """Chamberlin-Courant : le comité qui maximise la somme des utilités best-in-W.
    Copie pédagogique de la règle du carnet jumeau SC-09 (adaptations : signature
    uniforme (R, k, m) et retour du seul comité sous forme de tuple trié).
    À égalité de score, le comité lexicographiquement le plus petit est retenu."""
    if m is None:
        m = len(R[0])
    cands = sorted(set().union(*R))
    meilleur, meilleur_score = None, -1
    for W in combinations(cands, k):
        score = sum(score_borda(r, W, m) for r in R)
        if meilleur is None or score > meilleur_score or (score == meilleur_score and W < meilleur):
            meilleur, meilleur_score = W, score
    return tuple(sorted(meilleur))


def monroe(R, k, m=None):
    """Monroe (1995, stricte) : comité + affectation équilibrée (quotas [floor(n/k), ceil(n/k)]).
    Copie pédagogique de la règle du carnet jumeau SC-09 (adaptation : signature uniforme
    et retour du seul comité). Le score optimisé est la satisfaction Borda sur
    l'affectation assignée, pas le best-in-W (ce serait Chamberlin-Courant)."""
    if m is None:
        m = len(R[0])
    n = len(R)
    cands = sorted(set().union(*R))
    lo, hi = n // k, -(-n // k)

    def meilleure_affectation(W):
        quotas = {c: 0 for c in W}
        sigma = [None] * n
        best = (None, -1)

        def backtrack(i):
            nonlocal best
            if i == n:
                if all(quotas[c] >= lo for c in W):
                    score = sum(m - rang(R[j], sigma[j]) for j in range(n))
                    if score > best[1]:
                        best = (list(sigma), score)
                return
            for c in W:
                if quotas[c] < hi:
                    sigma[i] = c
                    quotas[c] += 1
                    backtrack(i + 1)
                    quotas[c] -= 1
                    sigma[i] = None

        backtrack(0)
        return best

    meilleur = None
    for W in combinations(cands, k):
        sigma, score = meilleure_affectation(W)
        if sigma is not None and (meilleur is None or score > meilleur[2]):
            meilleur = (tuple(sorted(W)), sigma, score)
    return meilleur[0] if meilleur is not None else ()


def stv(R, k, m=None):
    """STV au quota de Droop : élimination du plus bas, transfert du surplus à l'élection.
    Copie pédagogique de la règle du carnet jumeau SC-09, avec deux adaptations :
      (a) signature uniforme (R, k, m) et retour du comité sous forme de tuple trié ;
      (b) garantie de RÉSOLUTÉ : si le quota n'a pu être atteint par aucun candidat,
          le comité est complété par les SURVIVANTS (ceux qui n'ont pas été éliminés),
          classés par votes décroissants puis par ordre alphabétique — le théorème
          de Gibbard-Satterthwaite ne porte que sur des règles résolues.

    Convention (issue #21619) : l'élimination s'arrête dès qu'il ne reste plus assez
    de survivants pour occuper les sièges encore vacants, et la complétion se fait
    parmi les survivants. Compléter parmi les ÉLIMINÉS — tous à score nul — les
    départagerait alphabétiquement au lieu de les classer par popularité, et le
    résultat deviendrait instable : un vote parasite pourrait remplacer un élu par
    un candidat choisi par ordre alphabétique. Le comité rendu a donc toujours
    exactement k membres, et il est un ensemble canonique (trié)."""
    n = len(R)
    cands = sorted(set().union(*R))
    assert k <= len(cands), f"k={k} > m={len(cands)} : pas assez de candidats pour un comité de taille k"
    quota = n // (k + 1) + 1
    elus, vivants = [], set(cands)
    voix = Counter()
    for r in R:
        for c in r:
            if c in vivants:
                voix[c] += 1
                break
    while len(elus) < k:
        # ÉTAPE 1 : élection prioritaire — dès qu'un candidat atteint le quota.
        au_quota = [c for c in vivants if voix[c] >= quota]
        if au_quota:
            elu = min(au_quota, key=lambda c: (-voix[c], c))
            elus.append(elu)
            vivants.discard(elu)
            surplus = voix[elu] - quota
            if surplus > 0:
                for r in R:
                    passe_elu = False
                    for c in r:
                        if c == elu:
                            passe_elu = True
                            continue
                        if passe_elu and c in vivants and c not in elus:
                            voix[c] += surplus / voix[elu]
                            break
            voix[elu] = quota
            continue
        # ÉTAPE 2 : élimination du moins bien classé, tant qu'il reste assez de
        # survivants pour occuper les sièges encore vacants.
        if not vivants or len(vivants) <= k - len(elus):
            break
        elim = min(vivants, key=lambda c: (voix[c], c))
        porteurs = []
        for r in R:
            for c in r:
                if c in vivants or c in elus:
                    porteurs.append(c)
                    break
        vivants.discard(elim)
        for r, p in zip(R, porteurs):
            if p == elim:
                for c in r:
                    if c in vivants and c not in elus:
                        voix[c] += 1
                        break
        voix[elim] = 0
    # COMPLÉTION par les survivants (jamais parmi les éliminés, tous à score nul).
    if len(elus) < k:
        survivants = sorted(vivants, key=lambda c: (-voix[c], c))
        elus.extend(survivants[:k - len(elus)])
    assert len(elus) == k
    return tuple(sorted(elus))


REGLES = {
    "Pluralite (top-k)": pluralite,
    "Borda (top-k)": borda_topk,
    "Chamberlin-Courant": cc,
    "Monroe": monroe,
    "STV (Droop)": stv,
}

# Profil de démonstration : 4 votants, 3 candidats, comités de taille 1 et 2.
R_demo = [('a', 'b', 'c'), ('b', 'c', 'a'), ('c', 'a', 'b'), ('b', 'c', 'a')]
print("Profil de démonstration :", [''.join(r) for r in R_demo])
for nom, regle in REGLES.items():
    print(f"  {nom:<20} k=1 -> {regle(R_demo, 1, 3)}   k=2 -> {regle(R_demo, 2, 3)}")


Profil de démonstration : ['abc', 'bca', 'cab', 'bca']
  Pluralite (top-k)    k=1 -> ('b',)   k=2 -> ('b', 'a')
  Borda (top-k)        k=1 -> ('b',)   k=2 -> ('b', 'c')
  Chamberlin-Courant   k=1 -> ('b',)   k=2 -> ('a', 'b')
  Monroe               k=1 -> ('b',)   k=2 -> ('a', 'b')
  STV (Droop)          k=1 -> ('b',)   k=2 -> ('b', 'c')


## 4. Le détecteur de manipulation

Une **déviation gagnante** est un couple (profil, faux bulletin) tel que :

1. le faux bulletin diffère du bulletin vrai du votant ;
2. le comité change (au sens de l'**ensemble**) ;
3. l'utilité du votant pour le nouveau comité est **strictement supérieure** à son utilité pour l'ancien, calculée sur ses **préférences vraies**.

Le troisième point est essentiel : c'est le votant, et non le système, qui doit gagner. Un changement de comité qui n'améliore rien pour personne n'est pas une manipulation.

Le détecteur est **exhaustif** : il essaie tous les faux bulletins possibles pour chaque votant, et renvoie le **premier** trouvé. Une variante renvoie **toutes** les déviations gagnantes — c'est elle qui sert le witness de la section 4.


In [4]:
def detecte_violation(R, k, regle, m, faux_bulletins):
    """Première déviation gagnante trouvée pour ce profil, ou None.
    Le gain du votant est mesuré sur ses préférences VRAIES."""
    W = regle(R, k, m)
    for i in range(len(R)):
        for faux in faux_bulletins:
            if list(faux) == list(R[i]):
                continue
            R2 = list(R)
            R2[i] = faux
            W2 = regle(R2, k, m)
            if tuple(sorted(W2)) == tuple(sorted(W)):
                continue
            if utilite(R[i], W2, m) > utilite(R[i], W, m):
                return dict(votant=i, faux_bulletin=faux, comite_honnete=W,
                            comite_manipule=W2,
                            utilite_honnete=utilite(R[i], W, m),
                            utilite_manipulee=utilite(R[i], W2, m))
    return None


def manipulations(R, k, regle, m, faux_bulletins):
    """Toutes les déviations gagnantes du profil, tous votants confondus."""
    W = regle(R, k, m)
    out = []
    for i in range(len(R)):
        for faux in faux_bulletins:
            if list(faux) == list(R[i]):
                continue
            R2 = list(R)
            R2[i] = faux
            W2 = regle(R2, k, m)
            if tuple(sorted(W2)) == tuple(sorted(W)):
                continue
            if utilite(R[i], W2, m) > utilite(R[i], W, m):
                out.append(dict(votant=i, faux_bulletin=faux, comite_honnete=W,
                                comite_manipule=W2,
                                utilite_honnete=utilite(R[i], W, m),
                                utilite_manipulee=utilite(R[i], W2, m)))
    return out


faux_possibles = list(permutations(('a', 'b', 'c')))

print("Exemple 6.1 -- première déviation gagnante :")
print("  ", detecte_violation(R61, 1, pluralite, 3, faux_possibles))

print("\nExemple 6.1 -- toutes les déviations gagnantes :")
for d in manipulations(R61, 1, pluralite, 3, faux_possibles):
    print(f"  v{d['votant'] + 1} déclare {''.join(d['faux_bulletin'])} : "
          f"{d['comite_honnete']} -> {d['comite_manipule']}  "
          f"(utilité {d['utilite_honnete']} -> {d['utilite_manipulee']})")


Exemple 6.1 -- première déviation gagnante :
   {'votant': 1, 'faux_bulletin': ('c', 'a', 'b'), 'comite_honnete': ('a',), 'comite_manipule': ('c',), 'utilite_honnete': 0, 'utilite_manipulee': 1}

Exemple 6.1 -- toutes les déviations gagnantes :
  v2 déclare cab : ('a',) -> ('c',)  (utilité 0 -> 1)
  v2 déclare cba : ('a',) -> ('c',)  (utilité 0 -> 1)
  v3 déclare bac : ('a',) -> ('b',)  (utilité 0 -> 1)
  v3 déclare bca : ('a',) -> ('b',)  (utilité 0 -> 1)


### Lecture du résultat

Le profil de l'Exemple 6.1 est manipulable par **deux** votants, et non par une seule : la votante 2 (qui préfère `b`) et la votante 3 (qui préfère `c`) trouvent chacune un bulletin faux qui leur fait gagner un rang. Le manuel ne cite que la seconde.

Les **quatre** déviations gagnantes trouvées sont toutes de gain **0 → 1** :

- votante 2, qui déclare `c > a > b` : le comité devient `('c',)`, elle obtient son **deuxième** choix au lieu de son dernier ;
- votante 2, qui déclare `c > b > a` : même effet, autre bulletin ;
- votante 3, qui déclare `b > a > c` : le comité devient `('b',)`, elle obtient son deuxième choix ;
- votante 3, qui déclare `b > c > a` : c'est le témoin du manuel.

La structure est symétrique et mérite d'être relevée : **chaque votante exploite le premier choix de l'autre**. La votante 2 ne peut pas gagner en déclarant `b` — ce serait un match nul à trois, départagé par ordre alphabétique en sa faveur défavorable ; elle doit donc pousser `c`, le premier choix de la votante 3, pour emporter deux voix. Réciproquement, la votante 3 doit pousser `b`, le premier choix de la votante 2.

Retenons enfin que le gain est d'un seul rang. Le théorème de Gibbard-Satterthwaite ne dit rien des *montants* : il dira qu'une déviation existe, pas qu'elle vaut un rang ou dix.

## 5. Les trois branches du Théorème 6.2

Le texte du manuel (p. 128) :

> **Theorem 6.2 (Gibbard, 1973; Satterthwaite, 1975).** *Consider a (resolute) voting rule that is defined for some number m of alternatives with m ≥ 3, with no restrictions on the preference domain. Then, this rule must be at least one of the following:*
> 1. *dictatorial: there exists a single fixed voter whose most-preferred alternative is chosen for every profile;*
> 2. *imposing: there is at least one alternative that does not win under any profile;*
> 3. *manipulable (i.e., not strategyproof).*

Autrement dit : sur un domaine de préférences sans restriction et avec au moins trois candidats, **toute règle résolue est soit dictatoriale, soit imposante, soit manipulable**. Les deux premières branches sont inacceptables en pratique — une règle dictatoriale ignore tous les votants sauf un, une règle imposante écarte un candidat par principe — ; reste la troisième.

La section 5 **mesure** les deux premières branches sur l'espace exhaustif des 216 profils à trois votants et trois candidats, pour chacune des cinq règles. Si aucune n'est dictatoriale ni imposante, le théorème **force** la troisième : la règle est manipulable — et la section 6 en donnera le taux.


In [5]:
faux_possibles = list(permutations(('a', 'b', 'c')))
profils3 = [list(R) for R in product(faux_possibles, repeat=3)]

print(f"Espace exhaustif n=3, m=3 : {len(profils3)} profils")
print(f"{'Regle':<20}{'gagnants':<10}{'imposante':<12}{'dictatoriale':<14}branche 3")
print("-" * 76)
for nom, regle in REGLES.items():
    elus = set()
    for R in profils3:
        elus |= set(regle(R, 1, 3))
    imposante = len(elus) < 3
    dictatoriale = any(all(regle(R, 1, 3) == (R[v][0],) for R in profils3)
                       for v in range(3))
    branche3 = "manipulable" if not imposante and not dictatoriale else "-"
    print(f"{nom:<20}{''.join(sorted(elus)):<10}{str(imposante):<12}"
          f"{str(dictatoriale):<14}{branche3}")
print("\nBranche 3 = « manipulable » : elle est FORCÉE dès que les branches 1 et 2 sont fausses.")


Espace exhaustif n=3, m=3 : 216 profils
Regle               gagnants  imposante   dictatoriale  branche 3
----------------------------------------------------------------------------
Pluralite (top-k)   abc       False       False         manipulable
Borda (top-k)       abc       False       False         manipulable
Chamberlin-Courant  abc       False       False         manipulable
Monroe              abc       False       False         manipulable
STV (Droop)         abc       False       False         manipulable

Branche 3 = « manipulable » : elle est FORCÉE dès que les branches 1 et 2 sont fausses.


## 6. Fréquence de manipulation : énumération exhaustive

La manipulabilité d'un profil est un fait binaire ; ce qui intéresse l'agent de conception est **la fréquence** : sur quelle proportion des profils une déviation gagnante existe-t-elle ? Le carnet répond par **énumération exhaustive** — aucun échantillonnage, aucun profil oublié.

Espace de référence : `n = 3` votants, `m = 3` candidats, soit `6^3 = 216` profils (chaque votant peut classer les trois candidats de `3! = 6` façons). Pour chaque profil et chaque règle, le détecteur essaie les `5` faux bulletins de chaque votant : le coût est de l'ordre de `216 × 3 × 5` évaluations de règle par case du tableau.

Deux tailles de comité sont comparées : `k = 1` (élection d'un seul élu, le cas du théorème) et `k = 2` (comité, le cas multi-vainqueurs).


In [6]:
def enumere(regle, n, m, k, faux_bulletins):
    """Énumère tous les profils d'un espace et compte ceux qui sont manipulables."""
    total = viol = 0
    for R in product(faux_bulletins, repeat=n):
        R = list(R)
        total += 1
        if detecte_violation(R, k, regle, m, faux_bulletins) is not None:
            viol += 1
    return total, viol


print(f"{'Regle':<20}{'k=1':<20}{'k=2':<20}")
print("-" * 60)
for nom, regle in REGLES.items():
    cellules = []
    for k in (1, 2):
        total, viol = enumere(regle, 3, 3, k, faux_possibles)
        cellules.append(f"{viol:>4}/{total} ({100 * viol / total:5.1f} %)")
    print(f"{nom:<20}{cellules[0]:<20}{cellules[1]:<20}")

# GARDE : le STV rend toujours un comité de cardinal exactement k, et ce comité est
# un ensemble canonique (trié). On le vérifie sur TOUS les profils de l'espace
# exhaustif n = 3, m = 3 (216 profils), pour k = 1 et k = 2 — soit 432 comités.
n_comites_verifies = 0
for k_garde in (1, 2):
    for R_garde in product(faux_possibles, repeat=3):
        W_garde = stv(list(R_garde), k_garde)
        assert len(W_garde) == k_garde, (R_garde, k_garde, W_garde)
        assert list(W_garde) == sorted(W_garde), (R_garde, k_garde, W_garde)
        n_comites_verifies += 1
print(f"GARDE STV : {n_comites_verifies} comités vérifiés sur l'espace exhaustif "
      f"n=3, m=3, k=1 et k=2 — tous de cardinal k, tous triés.")


Regle               k=1                 k=2                 
------------------------------------------------------------
Pluralite (top-k)     36/216 ( 16.7 %)     0/216 (  0.0 %)  
Borda (top-k)         51/216 ( 23.6 %)    21/216 (  9.7 %)  
Chamberlin-Courant    51/216 ( 23.6 %)     0/216 (  0.0 %)  


Monroe                51/216 ( 23.6 %)     0/216 (  0.0 %)  

STV (Droop)           24/216 ( 11.1 %)     0/216 (  0.0 %)  
GARDE STV : 432 comités vérifiés sur l'espace exhaustif n=3, m=3, k=1 et k=2 — tous de cardinal k, tous triés.


### Lecture du résultat

Les fréquences sont mesurées, et elles contredisent une intuition répandue : **la règle la plus simple n'est pas la plus manipulable**.

En `k = 1`, la pluralité est manipulable sur **16,7 %** des profils (36/216) — moins que Borda, Chamberlin-Courant et Monroe, tous à **23,6 %** (51/216). Le STV est le plus robuste (**11,1 %**, 24/216) : exiger un quorum rend l'élu plus difficile à déloger, parce qu'un seul vote supplémentaire ne suffit pas à franchir le seuil.

En `k = 2`, l'écart se creuse et la structure change : la pluralité, Chamberlin-Courant, Monroe et le STV ne sont **jamais** manipulables sur cet espace (**0 %**), tandis que Borda l'est sur **9,7 %** (21/216). Avec trois candidats et deux élus, la marge est si étroite que la plupart des règles deviennent insensibles à la déclaration d'un votant : il faut que le comité **change** pour qu'il y ait gain, et changer un comité de deux éléments sur trois est rare.

Le résultat du STV à `k = 2` mérite une note de lecture, car il dépend d'une **convention de fin de scrutin** qu'il faut énoncer. Le STV procède par élimination : le candidat le moins bien classé est écarté et son vote transféré au premier candidat encore vivant. À force d'éliminer, on peut épuiser les candidats avant d'avoir pourvu tous les sièges — et il faut alors décider qui les occupe. Deux réponses sont concevables, et elles ne donnent pas le même résultat.

- **Compléter parmi les éliminés** : tous étant à score nul, ce tri revient en pratique à un **départage alphabétique**. Le comité dépendrait alors de l'ordre des lettres plutôt que de la popularité — et un vote parasite pourrait faire entrer un candidat par ordre alphabétique au lieu de son mérite. Le résultat n'est plus canonique.
- **Compléter parmi les survivants** : on cesse d'éliminer dès qu'il ne reste plus assez de survivants pour occuper les sièges vacants, et l'on complète par les survivants, classés par votes décroissants puis alphabétiquement. L'ordre d'élimination cesse de faire partie du résultat, et le comité a toujours exactement `k` membres.

C'est la seconde convention qui est appliquée ici ; l'alignement du carnet jumeau SC-09 sur cette convention est proposé par la PR #21632, encore ouverte à la date de cette version — tant qu'elle n'est pas fusionnée, les deux carnets peuvent différer sur cet espace. Elle rend le STV **stable** : le comité rendu est un ensemble canonique, caractérisé par sa taille `k` et son ordre de présentation trié — le départage alphabétique, convention de tout ce carnet, reste en vigueur à l'intérieur de cette procédure. La garde en fin de section vérifie sur les 216 profils de l'espace exhaustif, pour `k = 1` et `k = 2`, que le comité rendu a toujours exactement `k` membres et qu'il est trié.

Précision de lecture : ces taux valent pour les règles **telles qu'implémentées dans ce carnet** et pour **cet** espace de profils. « 0 % » signifie « aucune déviation gagnante trouvée sur 216 profils » — c'est une affirmation **finie**, jamais une preuve de strategy-proofness.

## 7. Un espace plus large : quatre candidats

L'espace à trois candidats est le plus petit qui contienne un exemple de manipulation, et le plus petit où les cinq règles se distinguent. Il a toutefois une limite structurelle : avec trois candidats et deux élus, le comité ne peut exclure qu'un seul candidat, et la marge de manœuvre d'un votant est étroite. Avec quatre candidats et trois votants, l'écart entre les premiers choix s'élargit, et les profils « fractionnés » — un candidat en tête d'un seul votant, deux autres en tête d'un seul votant chacun — deviennent possibles. C'est le terrain où la manipulation gagnante a le plus de chances d'apparaître.

On énumère donc `n = 3` votants et `m = 4` candidats, soit `24^3 = 13 824` profils — l'énumération reste exhaustive. Le nombre de profils croît d'un facteur 64 par rapport à l'espace précédent, et le nombre de faux bulletins par votant passe de `5` à `23` : les facteurs cités donnent 64 × 23/5 = 294,4 sur le nombre d'essais — ce facteur compte les évaluations de règle, pas un temps mesuré (les durées réelles sont dans les sorties de cellules). C'est le dernier espace que l'on énumère intégralement ; au-delà, il faudra échantillonner.


In [7]:
cands4 = ('a', 'b', 'c', 'd')
faux4 = list(permutations(cands4))
print(f"Espace exhaustif n=3, m=4 : {len(faux4) ** 3} profils")

for k in (1, 2):
    print(f"\nComité de taille k = {k}")
    for nom, regle in REGLES.items():
        t0 = time.monotonic()
        total, viol = enumere(regle, 3, 4, k, faux4)
        print(f"  {nom:<20} {viol:>6}/{total}  ({100 * viol / total:5.1f} %)"
              f"   [{time.monotonic() - t0:.1f}s]")


Espace exhaustif n=3, m=4 : 13824 profils

Comité de taille k = 1


  Pluralite (top-k)      4068/13824  ( 29.4 %)   [3.5s]


  Borda (top-k)          7080/13824  ( 51.2 %)   [3.7s]


  Chamberlin-Courant     7080/13824  ( 51.2 %)   [5.7s]


  Monroe                 7080/13824  ( 51.2 %)   [12.5s]


...

  Pluralite (top-k)       864/13824  (  6.2 %)   [4.2s]


  Borda (top-k)          3630/13824  ( 26.3 %)   [5.2s]


  Chamberlin-Courant     4512/13824  ( 32.6 %)   [11.4s]


  Monroe                 4512/13824  ( 32.6 %)   [64.3s]


  STV (Droop)             864/13824  (  6.2 %)   [10.2s]


### Lecture du résultat

L'élargissement de l'espace change tout, et pas dans le sens attendu.

En `k = 1`, les taux **doublent** : la pluralité passe de 16,7 % à **29,4 %**, Borda / Chamberlin-Courant / Monroe de 23,6 % à **51,2 %**, le STV de 11,1 % à **20,8 %**. Avec quatre candidats, il devient banal qu'un candidat soit en tête d'un seul votant, et un seul vote parasite suffit alors à déplacer l'élu.

En `k = 2`, le classement s'inverse :

| Règle | m = 3 | m = 4 |
|---|---|---|
| Pluralité (top-k) | 0,0 % | **6,2 %** |
| Borda (top-k) | 9,7 % | 26,3 % |
| Chamberlin-Courant | 0,0 % | **32,6 %** |
| Monroe | 0,0 % | **32,6 %** |
| STV (Droop) | 0,0 % | 6,2 % |

C'est le résultat le plus contre-intuitif du carnet : **les règles qui maximisent la satisfaction agrégée sont les plus manipulables**. La raison est structurelle : ces règles cherchent un **optimum**, et un optimum est instable par nature — un vote modifié suffit à déplacer le comité d'un cran et à faire gagner un rang à celui qui a menti. À l'inverse, une règle qui applique une **procédure** (STV) ou un **seuil** (pluralité) est moins déplaçable : son résultat est plus éloigné de la frontière qui sépare deux comités.

La ligne du STV mérite d'être lue avec la convention de fin de scrutin énoncée en section 6 : le comité est complété par les **survivants**, et l'ordre d'élimination ne fait pas partie du résultat. Le STV se place ainsi au niveau de la pluralité (6,2 %) comme règle la moins déplaçable, loin derrière les règles d'optimisation.

Une comparaison s'impose avec l'espace à trois candidats : le taux de Chamberlin-Courant passe de 0 % à 32,6 %. Le « 0 % » n'était donc pas une propriété de la règle, mais une propriété de l'**espace** : avec trois candidats et deux élus, la marge était insuffisante pour qu'un comité change. C'est la démonstration qu'un « 0 % » mesuré est un fait **borné**, jamais une propriété de la règle.

## 8. Z3 : chercher la déviation symboliquement

L'énumération exhaustive a un coût qui croît brutalement : au-delà de quatre candidats, on ne peut plus tout énumérer. La section 6.4 du manuel (p. 131, *Computational Hardness as a Barrier to Manipulation*) pose la question inverse — et plus féconde : si **trouver** une manipulation est difficile, alors la manipulation devient un obstacle de calcul, même si elle existe. Le manuel montre que, pour certaines règles et certains domaines, calculer une manipulation gagnante est PSPACE-difficile : la complexité devient alors une *barrière* à la manipulation.

Le carnet illustre cette idée sur Chamberlin-Courant : plutôt que d'énumérer les faux bulletins, on **pose la question au solveur** — « existe-t-il un faux bulletin du votant `i` qui déplace le comité et lui donne un rang de plus ? ». Le nombre de variables est fixe (une position par candidat), la question est arithmétique, et Z3 répond SAT ou UNSAT.

L'encodage doit porter le **départage** de la règle : le comité retenu est celui qui maximise la somme des utilités best-in-W et, à égalité, le comité lexicographiquement le plus petit. Deux conditions sont donc à encoder — le comité candidat est un **maximiseur** (son score est supérieur ou égal à celui de tout autre comité), et il est le **plus petit** parmi les maximiseurs (son score est strictement supérieur à celui de tout comité plus petit que lui).

Deux propriétés rendent cet encodage vérifiable :

- le **gain** du votant est calculé sur ses préférences **vraies** : il ne dépend pas du faux bulletin, c'est une constante du problème ;
- le comité candidat est **fixé pendant la résolution** : on boucle sur les comités possibles, et on demande au solveur s'il existe un faux bulletin qui fait de ce comité l'élu.


In [8]:
from z3 import Int, Solver, sat, Distinct, If, Sum, And


def deviation_cc_z3(R, k, m, i, W_honnete, restreint=False):
    """Déviation gagnante du votant i sous Chamberlin-Courant, cherchée symboliquement.

    Variables : p_c = position du candidat c dans le faux bulletin (1 = préféré).
    Un comité Wp est l'issue de CC sous le faux bulletin ssi :
      (a) score(Wp) >= score(Wq) pour tout autre comité Wq  (Wp est un maximiseur) ;
      (b) score(Wp) >  score(Wq) pour tout Wq < Wp au sens lexicographique
          (départage : à égalité de score, le comité le plus petit est retenu).
    Le gain du votant est mesuré sur son bulletin VRAI : c'est une constante."""
    cands = sorted(set().union(*R))
    utilite_honnete = utilite(R[i], W_honnete, m)
    for Wp in combinations(cands, k):
        Wp = tuple(sorted(Wp))
        if utilite(R[i], Wp, m) <= utilite_honnete:
            continue          # le votant n'y gagne rien : inutile de résoudre
        s = Solver()
        p = {c: Int(f"p_{c}") for c in cands}
        for c in cands:
            s.add(p[c] >= 1, p[c] <= m)
        s.add(Distinct([p[c] for c in cands]))
        if restreint:
            s.add(p[R[i][0]] == 1, p[R[i][1]] == 2)

        def utilite_expr(W):
            t = Int(f"t_{'_'.join(W)}")
            s.add(And([p[c] >= t for c in W]))
            s.add(Sum([If(p[c] == t, 1, 0) for c in W]) >= 1)
            return m - t

        def score(W):
            base = sum(score_borda(R[j], W, m) for j in range(len(R)) if j != i)
            return base + utilite_expr(W)

        for Wq in combinations(cands, k):
            Wq = tuple(sorted(Wq))
            if Wq == Wp:
                continue
            s.add(score(Wp) >= score(Wq))
            if Wq < Wp:
                s.add(score(Wp) > score(Wq))
        if s.check() == sat:
            model = s.model()
            faux = tuple(sorted(cands, key=lambda c: model[p[c]].as_long()))
            return faux, Wp
    return None, None


def verifie_violation(R, k, regle, m, faux, i):
    """Vérification en Python d'une déviation proposée : le gain est-il réel ?"""
    R2 = list(R)
    R2[i] = faux
    W = regle(R, k, m)
    W2 = regle(R2, k, m)
    if tuple(sorted(W2)) == tuple(sorted(W)):
        return None
    if utilite(R[i], W2, m) > utilite(R[i], W, m):
        return dict(comite_honnete=W, comite_manipule=W2,
                    utilite_honnete=utilite(R[i], W, m),
                    utilite_manipulee=utilite(R[i], W2, m))
    return None


# --- Contrôle positif : un profil où la force brute a déjà trouvé une déviation ---
R_pos = [('d', 'b', 'e', 'c', 'a'), ('b', 'd', 'c', 'a', 'e'),
         ('d', 'e', 'a', 'c', 'b'), ('e', 'd', 'c', 'b', 'a')]
k_pos, m_pos = 2, 5
W_h_pos = cc(R_pos, k_pos, m_pos)
print("Profil de contrôle : n=4, m=5, k=2")
print("  comité honnête (CC) :", W_h_pos)
for i in range(len(R_pos)):
    faux, Wp = deviation_cc_z3(R_pos, k_pos, m_pos, i, W_h_pos)
    if faux is None:
        print(f"  v{i + 1} : aucune déviation profitable (UNSAT)")
        continue
    ver = verifie_violation(R_pos, k_pos, cc, m_pos, faux, i)
    print(f"  v{i + 1} : Z3 -> {''.join(faux)} , comité {Wp}")
    print(f"         vérification Python : {ver}")

# --- Variante restreinte : le votant garde l'ordre de ses deux premiers choix ---
print("\nVariante restreinte : le votant garde l'ordre de ses deux premiers choix")
for i in range(len(R_pos)):
    faux, Wp = deviation_cc_z3(R_pos, k_pos, m_pos, i, W_h_pos, restreint=True)
    print(f"  v{i + 1} : {'SAT -> ' + ''.join(faux) if faux else 'UNSAT'}")

# --- Validation croisée : le solveur et la force brute concordent-ils ? ---
random.seed(20261020)
N = 200
brute_sat = z3_sat = concordances = 0
ecarts = []
t0 = time.monotonic()
for _ in range(N):
    n = random.randint(3, 4)
    m = random.randint(3, 4)
    k = 2
    cands = tuple(chr(ord('a') + j) for j in range(m))
    Rp = [tuple(random.sample(cands, m)) for _ in range(n)]
    faux_p = list(permutations(cands))
    W_h = cc(Rp, k, m)
    brute = detecte_violation(Rp, k, cc, m, faux_p)
    trouve = None
    for i in range(n):
        faux, Wp = deviation_cc_z3(Rp, k, m, i, W_h)
        if faux is not None:
            trouve = (i, faux, Wp)
            break
    if brute is not None:
        brute_sat += 1
    if trouve is not None:
        z3_sat += 1
        if verifie_violation(Rp, k, cc, m, trouve[1], trouve[0]) is None:
            ecarts.append((Rp, "Z3 propose une déviation non vérifiée", trouve))
    if (brute is None) == (trouve is None):
        concordances += 1
    else:
        ecarts.append((Rp, brute, trouve))

print(f"\nValidation croisée sur {N} profils aléatoires "
      f"(n in [3,4], m in [3,4], k=2) :")
print(f"  force brute : {brute_sat} profils manipulables")
print(f"  Z3          : {z3_sat} profils manipulables")
print(f"  concordance : {concordances}/{N}   écarts : {len(ecarts)}"
      f"   [{time.monotonic() - t0:.1f}s]")
for e in ecarts[:5]:
    print("   ÉCART :", [''.join(r) for r in e[0]], e[1], e[2])


Profil de contrôle : n=4, m=5, k=2
  comité honnête (CC) : ('b', 'd')
  v1 : aucune déviation profitable (UNSAT)
  v2 : aucune déviation profitable (UNSAT)
  v3 : aucune déviation profitable (UNSAT)
  v4 : Z3 -> eabcd , comité ('d', 'e')
         vérification Python : {'comite_honnete': ('b', 'd'), 'comite_manipule': ('d', 'e'), 'utilite_honnete': 3, 'utilite_manipulee': 4}

Variante restreinte : le votant garde l'ordre de ses deux premiers choix
  v1 : UNSAT
  v2 : UNSAT
  v3 : UNSAT


  v4 : UNSAT



Validation croisée sur 200 profils aléatoires (n in [3,4], m in [3,4], k=2) :
  force brute : 39 profils manipulables
  Z3          : 39 profils manipulables
  concordance : 200/200   écarts : 0   [2.1s]


### Lecture du résultat

Le contrôle positif fonctionne comme il le doit : sur ce profil de quatre votants et cinq candidats, avec `k = 2`, le comité honnête est `('b', 'd')` ; les votants 1, 2 et 3 n'ont **aucune** déviation profitable, et le votant 4 en a une — le faux bulletin `eabcd` déplace le comité vers `('d', 'e')` et fait passer son utilité de 3 à 4. Chaque déviation proposée par le solveur est ensuite **re-vérifiée en Python** : le gain annoncé est réel, mesuré sur les préférences vraies.

La **variante restreinte** donne un résultat plus instructif qu'un simple « ça marche encore ». En imposant au votant de conserver l'ordre de ses **deux** premiers choix, la réponse devient **UNSAT pour les quatre votants** : la déviation du votant 4 (`eabcd` contre son bulletin vrai `edcba`) consiste précisément à remplacer son **deuxième** choix. Autrement dit, sur ce profil, la manipulation n'est pas un simple vote stratégique — un vote qui désigne un autre premier choix — : elle exige de **réordonner** le bulletin, et le gain du votant ne se trouve pas dans le choix du premier candidat.

La validation croisée confirme la correction de l'encodage : sur 200 profils aléatoires (`n` et `m` de 3 ou 4, `k = 2`), la force brute et Z3 ont trouvé la même chose sur les 39 profils manipulables, la concordance est de **200/200** et les **écarts sont au nombre de 0**. La durée du calcul est affichée dans la sortie de la cellule : elle dépend de la machine et ne constitue pas un résultat.

Deux enseignements pour la conception de systèmes de vote :

- **chercher** une manipulation est un problème de satisfiabilité à nombre de variables fixe (une position par candidat) : l'encodage ci-dessus est polynomial en la taille du profil, là où l'énumération est exponentielle. C'est l'argument du manuel : quand la recherche de la manipulation devient coûteuse, elle cesse d'être praticable, et la règle gagne une barrière de calcul — sans pour autant cesser d'être manipulable ;
- mais la **décision** « ce profil est-il manipulable ? » ne se délégue pas au solveur seul : l'encodage doit porter le départage de la règle, et toute erreur d'encodage se paie en faux positifs. La validation croisée fait donc partie de la méthode, pas d'un supplément.

## 9. Exercices

Les trois exercices sont à réaliser dans les cellules de code qui suivent. Ils exploitent le matériel de ce carnet : les cinq règles, le détecteur, et les espaces énumérés.


### Exercice 1 — satisfaction moyenne d'un comité

Écrire une fonction `utilite_moyenne(R, W, m)` qui renvoie la satisfaction moyenne des votants pour le comité `W`.

Puis vérifier sur `R_demo` (4 votants, 3 candidats) que le comité renvoyé par `cc(R_demo, 2, 3)` est bien celui qui **maximise** cette quantité parmi tous les comités de taille 2 — en comparant explicitement les trois comités possibles.

*Indice* : `combinations(cands, 2)` énumère les comités de deux éléments.


In [9]:
# Exercice 1 -- a completer
# def utilite_moyenne(R, W, m):
#     ...
# comparer les trois comites de taille 2 sur R_demo et conclure.
pass


### Exercice 2 — comparer la robustesse de deux règles

Écrire une fonction `frequence_manipulation(regle, n, m, k)` qui renvoie le nombre de profils manipulables et le nombre total de profils d'un espace (réutiliser `enumere`).

Puis, sur l'espace `n = 3`, `m = 3`, `k = 1`, comparer la pluralité et le STV, et vérifier que la seconde est **moins** manipulable que la première.

*Indice* : le nombre de faux bulletins possibles pour un profil à `m` candidats est `m! - 1`.


In [10]:
# Exercice 2 -- a completer
# def frequence_manipulation(regle, n, m, k):
#     ...
# comparer pluralite et stv sur l'espace n=3, m=3, k=1.
pass


### Exercice 3 — les déviations d'un votant

Écrire une fonction `deviations_votant(R, i, k, regle, m, faux_bulletins)` qui renvoie **toutes** les déviations gainantes imputables au votant `i` seul.

L'appliquer au profil de l'Exemple 6.1 (`R61`) pour la votante 2, puis vérifier que cette votante dispose de **deux** déviations gainantes : `c > a > b` et `c > b > a`.

Constater alors que **ces deux déviations modifient toutes deux son premier choix** (de `b` à `c`), et expliquer pourquoi, **sous la pluralité**, aucune déviation qui **conserve** le premier choix ne peut être gagnante sur ce profil : le scrutin y est un match nul à trois, si bien que tout bulletin qui laisse la première position inchangée laisse les décomptes inchangés — donc ne peut pas déplacer le comité, ni donc améliorer l'utilité du votant.

*Indice* : la pluralité ne compte que les premières positions ; le comité ne peut changer que si un décompte change.


In [11]:
# Exercice 3 -- a completer
# def deviations_votant(R, i, k, regle, m, faux_bulletins):
#     ...
# appliquer a R61 pour la votante 2, puis expliquer.
pass


## 10. Conclusion

Ce carnet a fait quatre choses, chacune mesurée.

**1. Un témoin reproduit, puis décomposé.** L'Exemple 6.1 du manuel est reconstitué à l'identique — match nul à trois, départage alphabétique, gain d'un rang pour la votante 3. L'automatisation du détecteur révèle ensuite que ce profil admet **quatre** déviations gainantes, et non une seule : deux pour la votante 2 (que le manuel n'évoque pas), deux pour la votante 3. La structure est symétrique — chaque votante exploite le premier choix de l'autre.

**2. Le théorème de Gibbard-Satterthwaite, mesuré.** Sur l'espace exhaustif des 216 profils à trois votants et trois candidats, aucune des cinq règles n'est dictatoriale ni imposante. Les deux branches « acceptables » du Théorème 6.2 sont donc closes, et la troisième — *manipulable* — est **forcée**. Le théorème n'est pas seulement cité : il est vérifié sur un espace fini, et les quatre déviations de la section 4 en sont la démonstration constructive.

**3. Une fréquence qui dépend de l'espace, pas seulement de la règle.** Chamberlin-Courant passe de 0 % de profils manipulables en `m = 3, k = 2` à 32,6 % en `m = 4, k = 2`. Un « 0 % » mesuré est un fait borné, jamais une propriété de la règle. Et le classement des règles s'inverse entre les deux espaces : les règles qui maximisent une satisfaction agrégée (Chamberlin-Courant, Monroe) sont les plus manipulables, parce qu'un optimum est instable ; la pluralité et le STV, qui appliquent un seuil ou une procédure, le sont moins.

**4. La recherche symbolique, confrontée à l'énumération.** L'encodage Z3 de la déviation gagnante sous Chamberlin-Courant est polynomial en la taille du profil, là où l'énumération est exponentielle ; sa correction est vérifiée par agreement complet avec la force brute sur 200 profils aléatoires, et chaque déviation proposée par le solveur est re-vérifiée en Python sur les préférences vraies. C'est la « barrière de calcul » du manuel, montrée sur un encodage exécutable.

La suite naturelle serait d'encoder la même question pour les autres quatre règles, puis de mesurer le coût de la recherche de la manipulation pour des profils plus grands — c'est-à-dire de quantifier la barrière de calcul elle-même.

## Références

- **Brandt, G., Conitzer, V., Endriss, N., Lang, J.** (2016). *Handbook of Computational Social Choice*. Cambridge University Press. **Chapitre 6, « Barriers to Manipulation in Voting »** (p. 127-145) : §6.1 *Introduction* (p. 127, Exemple 6.1), §6.2 *Gibbard-Satterthwaite and Its Implications* (p. 128, Théorème 6.2), §6.3 *Noncomputational Avenues around Gibbard-Satterthwaite* (p. 129), §6.4 *Computational Hardness as a Barrier to Manipulation* (p. 131), §6.5 *Can Manipulation Be Hard Most of the Time?* (p. 139). **Chapitre 9, « Voting in Combinatorial Domains »** (Lang & Xia, p. 197-222), §9.3 *Approaches Based on Completion Principles* (p. 204-215) : cadre de la représentation proportionnelle (Chamberlin-Courant, Monroe) et fonctions de *misrepresentation*. **Chapitre 2, §2.8** *Strategyproofness: Impossibilities* (p. 46).
  - Exemplaire archivé (jamais copié dans le dépôt public) : `G:\Mon Drive\MyIA\IA\Bibliographie IA\GameTheory\2016 - Brandt Conitzer Endriss Lang - Handbook of Computational Social Choice.pdf`
- **Gibbard, A.** (1973) et **Satterthwaite, F.** (1975) — théorème de manipulabilité, repris au Théorème 6.2.
- **Monroe, H.** (1995) — *Free Choice or Compromise: An Introduction to Monroe's Rule*, Political Analysis.
- **Chamberlin, N. et Courant, J.** (1943) — la règle de représentation proportionnelle « top-choice ».

## Carnets jumeaux de la série

- `05-Gibbard-Satterthwaite.ipynb` — le théorème d'impossibilité, côté axiomes.
- `09-Committees-STV-Monroe-ChamberlinCourant.ipynb` — les règles à comité, côté règles.
- `08-Kemeny-RankedPairs-Dodgson.ipynb` — les règles fondées sur la majorité (Kemeny, ranked pairs, Dodgson), côté calcul.
